In [ ]:

from dotenv import load_dotenv
import os
from langchain_core.tools import tool
import json
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
import langchain_google_genai
from random import randint
import mysql.connector
from connections import get_product,get_connection
from scheam import (Item, Delivery, Accept_Offer, Reject, 
                    Request_Offer, Response_Offer,Availability_Request, 
                    Availability_Response)
from fastmcp import FastMCP
from connections import get_connection, get_product



In [ ]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

In [ ]:
SYSTEM_PROMPT="""
Jesteś drugą hurtownią z ID H2 w łańcuchu dostaw. Twoim zadaniem jest przyjmowanie zamówień od producenta i przekazywanie ich do dwóch restauracji R1 i R2.
Na stanie masz 11 produktów flour, passata, mozzarella, parmigiano reggiano, burrata, buffala, prosciutto cotto, prosciutto crudo, arugula, lamb's lettuce, salami. 
Przy sprzedazy produktów, należy pamiętać o maksymalnych ilościach dostępnych na stanie, nie jesteś w stanie sprzedać więcej niż jest dostępne. Ponadto sprzedawać możesz tylko produkty, które są dostępne w hurtowni, w liczbach naturalnych . Nie możesz sprzedawać produktów, których nie masz na stanie.
Zawsze korzystaj z dostępnych narzędzi, aby badać stan faktyczny hurtowni i podejmować decyzje. Odpowiadaj rzeczowo i precyzyjnie w języku polskim
"""

In [ ]:
@tool
def stock_info(product:str) -> Item:
    """
    Returns stock information about a specific product.
    """
    product= get_product(product)

    if product is None:
        return Item(name=product["name"],
                    quantity=0, price=0)

    return Item(name=product["name"],
        quantity=product["quantity"],
        price=product["price"])

@tool
def products_status() -> list[Item]:
    """
    Returns the status of all products in the warehouse.
    """
    connection = get_connection()
    try:
        cursor = connection.cursor(dictionary=True)
        cursor.execute(
            "SELECT name, quantity, price FROM warehouse"
        )
        rows = cursor.fetchall()
        return [ Item(
            name=item["name"],
            quantity=item["quantity"],
            price=item["price"])
            for item in rows]
    finally:
        cursor.close()
        connection.close()

@tool
def get_proposal(product:Item) -> Request_Offer:
    """
    Prepare request to buy specific product.
    """
    if product["name"] is None:
        return stock_info(product.name)
    return Request_Offer(
        sender_id="P1",
        reciver_id="H2",
        message_type="CALL_FOR_PROPOSAL",
        item=product)
    
@tool
def accept_offer_from_producer(Proposal:Response_Offer) -> Accept_Offer:
    """
    Accepts an offer from the producer. Checks if the warehouse is able to purchase it.
    """
    quantity=Proposal.item.quantity
    price=Proposal.item.price
    total_cost=quantity*price
    connection=get_connection()
    try:
        cursor = connection.cursor(dictionary=True)
        cursor.execute("SELECT ballance FROM wallet_warehouse2 ORDER BY id DESC LIMIT 1")
        row = cursor.fetchone()
        current_balance = row["ballance"]
        
        if current_balance < total_cost:
            return Reject(
                sender_id="P1",
                receiver="H2",
                message_type="REJECT_PROPOSAL",
                item=Item(
                    name=Proposal.item.name,
                    quantity=quantity,
                    price=price))   
    finally:
        cursor.close()
        connection.close()

    return Accept_Offer(
        sender_id="P1",
        receiver="H2",
        message_type="ACCEPT_PROPOSAL",
        item=Item(
            name=Proposal.item.name,
            quantity=quantity,
            price=price),
        total_cost=total_cost)


In [ ]:
tools = [stock_info, products_status, get_proposal, accept_offer_from_producer]

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    system_prompt=SYSTEM_PROMPT,
    tools=tools,
    checkpointer=InMemorySaver(),
)
config: RunnableConfig={"configurable":{"thread_id":"1"}}

In [ ]:
mcp = FastMCP("Warehouse H2")

# Warehouse as a seller agent MCP tools
# Step 1 Agent checks availability of the product in the warehouse
@mcp.tool
def check_availability(Availability_Request: Availability_Request) -> Availability_Response:
    """
    Checks whether the requested quantity of a product is available.
    """
    item = get_product(Availability_Request.item.name)

    if item is None:
        return Availability_Response(
            sender_id="H2",
            receiver_id=Availability_Request.sender_id,
            message_type="AVAILABILITY_RESPONSE",
            item=Availability_Request.item,
            is_available=False,
            available_quantity=0
        )
    
    if item["quantity"] <= 0:
        return Availability_Response(
            sender_id="H2",
            receiver_id=Availability_Request.sender_id,
            message_type="AVAILABILITY_RESPONSE",
            item=Availability_Request.item,
            is_available=False,
            available_quantity=item["quantity"])
    
    quantity = item["quantity"]

    if Availability_Request.item.quantity > quantity:
        return Availability_Response(
            sender_id="H2",
            receiver_id=Availability_Request.sender_id,
            message_type="AVAILABILITY_RESPONSE",
            item=Availability_Request.item,
            is_available=False,
            available_quantity=quantity)

    return Availability_Response(
        sender_id="H2",
        receiver_id=Availability_Request.sender_id,
        message_type="AVAILABILITY_RESPONSE",
        item=Availability_Request.item,
        is_available=True,
        available_quantity=quantity)

# Step 2 Warehouse responses to an restaurant's offer
@mcp.tool
def request_offer(Request_Offer: Request_Offer) -> Response_Offer:
    """
    Response to an offer for a specific product and quantity to restaurants.
    """
    item = get_product(Request_Offer.item.name)
    if item is None:
        return Response_Offer(
            sender_id="H2",
            receiver_id=Request_Offer.sender_id,
            message_type="PROPOSAL",
            item=Item(
                name=Request_Offer.item.name,
                quantity=Request_Offer.item.quantity,
                price=Request_Offer.item.price),
            total_cost=0.0)
    if Request_Offer.item.quantity <= 0:
        return Response_Offer(
            sender_id="H2",
            receiver_id=Request_Offer.sender_id,
            message_type="PROPOSAL",
            item=Item(
                name=Request_Offer.item.name,
                quantity=Request_Offer.item.quantity,
                price=Request_Offer.item.price
            ),
            total_cost=0.0)

    return Response_Offer(
        sender_id="H2",
        receiver_id=Request_Offer.sender_id,
        message_type="PROPOSAL",
        item=Item(
            name=Request_Offer.item.name,
            quantity=Request_Offer.item.quantity,
            price=item["price"]),
        total_cost=Request_Offer.item.quantity * item["price"])

# Step 4 Agent finalizes the order 
@mcp.tool
def accept_offer(Accept_Offer: Accept_Offer) -> Accept_Offer:
    """
    Sales a product from the warehouse, finalizing the order. 
    """
    product = get_product(Accept_Offer.item.name)
    if product is None or Accept_Offer.item.quantity > product["quantity"]:
        return Reject(
            sender_id="H2",
            receiver_id=Accept_Offer.receiver_id,
            message_type="REJECT_PROPOSAL",
            item=Item(
                name=Accept_Offer.item.name,
                quantity=Accept_Offer.item.quantity,
                price=Accept_Offer.item.price))

    return Accept_Offer(
            sender_id="H2",
            receiver_id=Accept_Offer.receiver_id,
            message_type="ACCEPT_PROPOSAL",
            item=Item(
                name=Accept_Offer.item.name,
                quantity=Accept_Offer.item.quantity,
                price=Accept_Offer.item.price),
            total_cost=Accept_Offer.item.quantity * Accept_Offer.item.price)


# Step 5 Warehouse delivers products and agent udates the warehouse stock
@mcp.tool
def receive_delivery(Accept_Offer: Accept_Offer) -> Delivery:
    """
    Updates the warehouse stock based on sold products.
    """
    connection = get_connection()
    name=Accept_Offer.item.name
    item=get_product(name)
    quantity=Accept_Offer.item.quantity
    total_cost=quantity*item["price"]
    try:
        cursor = connection.cursor()
        cursor.execute(
            """
            UPDATE warehouse2
            SET quantity = quantity - %s
            WHERE name = %s AND quantity >= %s
            """,
            (quantity,name, quantity)
            )
        connection.commit()
        cursor.execute(
            """
            INSERT INTO wallet_warehouse2 (sender_id,receiver_id,type,ballance)
            SELECT %s, 'H2', 'INCOME', ballance + %s
            FROM wallet_warehouse2
            ORDER BY id DESC LIMIT 1
            """, (Accept_Offer.sender_id,total_cost)
        )
        connection.commit()
    finally:
        cursor.close()
        connection.close()
    return Delivery(
        sender_id="H2",
        receiver_id=Accept_Offer.receiver_id,
        message_type="DELIVERY",
        item=Item(
            name=Accept_Offer.item.name,
            quantity=Accept_Offer.item.quantity,
            price=Accept_Offer.item.price),
        total_cost=Accept_Offer.item.quantity * Accept_Offer.item.price)

# Warehouse as a buyer agent
# Step 5 Agent updates products after new purchase
@mcp.tool
def receive_delivery_from_producer(Delivery:Delivery):
    '''
    Updates the warehouse stock based on the received products from the producer.
    '''
    product = get_product(Delivery.item.name)
    bought_quantity=Delivery.item.quantity
    price=Delivery.item.price
    total_cost = bought_quantity * price
    connection=get_connection()
    try:
        cursor = connection.cursor(dictionary=True)
        cursor.execute(
            """
            UPDATE warehouse2
            SET quantity = quantity + %s
            WHERE name = %s
            """,
            (bought_quantity, product["name"]))
        connection.commit()
        cursor.execute(
            """
            INSERT INTO wallet_warehouse2 (sender_id,receiver_id,type,ballance)
            SELECT "H2", %s, "EXPENSE", ballance - %s
            FROM wallet_warehouse2
            ORDER BY id DESC LIMIT 1
            """, (Delivery.sender_id,total_cost)
            )
        connection.commit()
    finally:
        cursor.close()
        connection.close()